# DRZEWO DECYZYJNE DLA <span style="color:red; font-size:26pt; font-family:Georgia;">PIMA INDIANS</span> DIABETES
## Dane bez nagłówków + obsługa zer + optymalizacja modelu

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# NAZWY KOLUMN

In [2]:
column_names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

# WCZYTANIE DANYCH BEZ NAGŁÓWKÓW

In [3]:
df = pd.read_csv(
    "diabetes_pima.csv",
    header=None,
    names=column_names
)

print("\nPierwsze rekordy:")
print(df.head())

print("\nInformacje o danych:")
print(df.info())

print("\nStatystyki opisowe:")
print(df.describe())

print("\nRozkład klas:")
print(df["Outcome"].value_counts())


Pierwsze rekordy:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Informacje o danych:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    i

# KOLUMNY, W KTÓRYCH ZERO OZNACZA PRAWDOPODOBNIE BRAK DANYCH

In [4]:
zero_as_missing_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

print("\nLiczba zer przed czyszczeniem:")

for col in zero_as_missing_columns:
    zero_count = (df[col] == 0).sum()
    print(f"{col}: {zero_count}")


Liczba zer przed czyszczeniem:
Glucose: 5
BloodPressure: 35
SkinThickness: 227
Insulin: 374
BMI: 11


# ZAMIANA PODEJRZANYCH ZER NA NaN

In [5]:
df_clean = df.copy()

for col in zero_as_missing_columns:
    df_clean[col] = df_clean[col].replace(0, np.nan)

print("\nLiczba braków danych po zamianie zer na NaN:")
print(df_clean.isna().sum())


Liczba braków danych po zamianie zer na NaN:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


# PODZIAŁ NA CECHY X I ETYKIETĘ y

In [6]:
X = df_clean.drop("Outcome", axis=1)
y = df_clean["Outcome"]

# PODZIAŁ NA DANE TRENINGOWE I TESTOWE

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# PIPELINE: IMPUTACJA BRAKÓW + DRZEWO DECYZYJNE

In [8]:
pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer()),
        ("model", DecisionTreeClassifier(random_state=42))
    ]
)